# DAIOE SSYK2012 Workflow

Organized end-to-end flow for preparing DAIOE + SCB SSYK data.


## 1) Setup Paths and Data Sources

Define imports, workspace paths, and source locations.


In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DAIOE_SOURCE: str = (
    "https://raw.githubusercontent.com/joseph-data/07_translate_ssyk/main/"
    "03_translated_files/daioe_ssyk2012_translated.csv"
)

SCB_SOURCE: str = (
        "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_years/daioe_pull/"
        "data/processed/ssyk12_aggregated_ssyk4_to_ssyk1.parquet"
)

SSYK12_MIN_YEAR: int = 2014                               # first year of SSYK 2012 publication in AM0208
SSYK4_CODE_LEN: int = 4                                   # number of digits in a SSYK4 code
QUINTILE_BOUNDS: tuple[int, int, int, int] = (20, 40, 60, 80)  # thresholds for 1-5 Level Exposure bins

## 2) Load DAIOE and SCB Lazily

Read source files as `LazyFrame` objects for efficient transformations.


In [2]:
daioe_lazy_lf = pl.scan_csv(
    DAIOE_SOURCE,
)

scb_lazy_lf = pl.scan_parquet(
    SCB_SOURCE,
)

## 3) Define Utility Helpers

Create helper functions used for lightweight pipeline inspection.


In [3]:
def inspect_lazy(lf: pl.LazyFrame) -> None:
    """
    Print the shape of a Polars LazyFrame in a memory-efficient manner.

    This function computes the number of rows using a lazy row-count
    aggregation (`pl.len()`) and retrieves the number of columns from
    the resolved schema without materializing the full dataset.

    Parameters
    ----------
    lf : pl.LazyFrame
        The LazyFrame to inspect.

    Notes
    -----
    - The row count triggers execution of the lazy query plan,
      but avoids collecting all columns into memory.
    - The column count is obtained from the schema metadata and
      does not require data materialization.
    - Intended for debugging and validation of large lazy pipelines.

    """
    n_rows = lf.select(pl.len()).collect().item()
    n_cols = len(lf.collect_schema())
    print(f"Rows: {n_rows:,}")
    print(f"Columns: {n_cols}")


## 4) Quick Sanity Checks and Early Military Removal

Preview both sources and remove code-0 military rows early from DAIOE and SCB.


In [4]:
print(daioe_lazy_lf.head(5).collect())

shape: (5, 27)
┌────────────┬──────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ ssyk2012_4 ┆ year ┆ daioe_alla ┆ daioe_stra ┆ … ┆ pctl_rank_ ┆ ssyk2012_ ┆ ssyk2012_ ┆ ssyk2012_ │
│ ---        ┆ ---  ┆ pps        ┆ tgames     ┆   ┆ genai      ┆ 1         ┆ 2         ┆ 3         │
│ str        ┆ i64  ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆      ┆ f64        ┆ f64        ┆   ┆ f64        ┆ str       ┆ str       ┆ str       │
╞════════════╪══════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 0110 Commi ┆ 2010 ┆ NaN        ┆ NaN        ┆ … ┆ NaN        ┆ 0 Armed   ┆ 01        ┆ 011 Commi │
│ ssioned    ┆      ┆            ┆            ┆   ┆            ┆ forces    ┆ Officers  ┆ ssioned   │
│ armed      ┆      ┆            ┆            ┆   ┆            ┆ occupatio ┆           ┆ armed     │
│ forces…    ┆      ┆            ┆            ┆   ┆            ┆ ns        ┆

In [5]:
print(scb_lazy_lf.head(5).collect())

shape: (5, 7)
┌───────┬───────────┬───────┬───────┬──────┬───────┬─────────────────────────────────┐
│ level ┆ ssyk_code ┆ age   ┆ sex   ┆ year ┆ count ┆ occupation                      │
│ ---   ┆ ---       ┆ ---   ┆ ---   ┆ ---  ┆ ---   ┆ ---                             │
│ str   ┆ str       ┆ str   ┆ str   ┆ i64  ┆ i64   ┆ str                             │
╞═══════╪═══════════╪═══════╪═══════╪══════╪═══════╪═════════════════════════════════╡
│ SSYK4 ┆ 3111      ┆ 25-29 ┆ men   ┆ 2021 ┆ 495   ┆ Industrial, logistics and prod… │
│ SSYK4 ┆ 2342      ┆ 40-44 ┆ women ┆ 2018 ┆ 1552  ┆ Recreation leaders              │
│ SSYK4 ┆ 1512      ┆ 60-64 ┆ men   ┆ 2022 ┆ 189   ┆ Department and unit managers i… │
│ SSYK4 ┆ 5112      ┆ 30-34 ┆ men   ┆ 2016 ┆ 286   ┆ Train attendants, persons resp… │
│ SSYK4 ┆ 7224      ┆ 60-64 ┆ women ┆ 2014 ┆ 22    ┆ Metal polishers, wheel grinder… │
└───────┴───────────┴───────┴───────┴──────┴───────┴─────────────────────────────────┘


In [6]:
# daioe_lazy_lf.collect_schema()

In [7]:
## Removed Military Personnel

scb_lazy_lf = scb_lazy_lf.filter(
    pl.col("ssyk_code").str.starts_with("0").not_(),
)

daioe_lazy_lf = daioe_lazy_lf.filter(
    pl.col("ssyk2012_4").str.starts_with("0").not_(),
)

## 4b) Add Age Groups to SCB

Map raw SCB age bands to career-stage labels before any downstream joins.

In [ ]:
age_map = {
    "16-24": "Early Career 1 (16-24)",
    "25-29": "Early Career 2 (25-29)",
    "30-34": "Developing (30-34)",
    "35-39": "Mid-Career 1 (35-39)",
    "40-44": "Mid-Career 1 (40-44)",
    "45-49": "Mid-Career 2 (45-49)",
    "50-54": "Senior (50+)",
    "55-59": "Senior (50+)",
    "60-64": "Senior (50+)",
    "065-69": "Senior (50+)",
}

scb_lazy_lf = scb_lazy_lf.with_columns(
    pl.col("age").replace(age_map).alias("age_group"),
)

## 5) Derive SSYK Levels in DAIOE

Split DAIOE SSYK4 into SSYK1-4 and keep SSYK2012-era years.


In [9]:
daioe_lazy_lf_ssyk12 = (
    daioe_lazy_lf\
    .with_columns([
    pl.col("ssyk2012_4").str.slice(0, 1).alias("code_1"),
    pl.col("ssyk2012_4").str.slice(0, 2).alias("code_2"),
    pl.col("ssyk2012_4").str.slice(0, 3).alias("code_3"),
    pl.col("ssyk2012_4").str.slice(0, 4).alias("code_4"),
])\
    .drop(pl.col("^ssyk2012.*$"))\
        .filter(pl.col("year") >= SSYK12_MIN_YEAR) ## The Year stretch from the first SSYK12 publication
)

## 6) Align DAIOE Years to SCB Coverage

Extend DAIOE series forward when SCB has later years.


In [10]:
## Here I extend the years to Latest according to the pulled SCB data (2024, yearly)

base = daioe_lazy_lf_ssyk12

daioe_max = base.select(pl.max("year")).collect().item()
scb_max   = scb_lazy_lf.select(pl.max("year")).collect().item()

missing = list(range(daioe_max + 1, scb_max + 1))

daioe_lazy_lf_extended = (
    base
    if not missing
    else pl.concat(
        [
            base,
            base
            .filter(pl.col("year") == daioe_max)
            .drop("year")
            .join(pl.LazyFrame({"year": missing}), how="cross")
            .select(base.collect_schema().names()),  # ensure same column order/schema
        ],
        how="vertical",
    )
)



In [11]:
inspect_lazy(daioe_lazy_lf_extended)


Rows: 4,686
Columns: 27


## 7) Build SCB SSYK4 Employment Counts

Aggregate SCB counts at year + 4-digit SSYK level.


In [12]:
scb_lazy_lf_level4 = (
    scb_lazy_lf
        .filter(pl.col("ssyk_code").str.len_chars() == SSYK4_CODE_LEN)
        .group_by(["year", "ssyk_code"])
        .agg(pl.col("count").sum().alias("total_count"))
)

In [13]:
inspect_lazy(scb_lazy_lf_level4)

Rows: 4,686
Columns: 3


## 8) Merge DAIOE with SCB SSYK4 Counts

Join by `year` and 4-digit code, then inspect merged coverage.


In [14]:
daioe_lazy_lf_extended.head(5).collect()

year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""1""","""11""","""111""","""1111"""
2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""1""","""11""","""111""","""1111"""
2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""1""","""11""","""111""","""1111"""
2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""1""","""11""","""111""","""1111"""
2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""1""","""11""","""111""","""1111"""


In [15]:
scb_lazy_lf_level4.head(5).collect()

year,ssyk_code,total_count
i64,str,i64
2021,"""7534""",358
2023,"""2283""",3201
2018,"""8116""",681
2021,"""6121""",6685
2020,"""2513""",3773


In [16]:
daioe_scb_years = daioe_lazy_lf_extended\
    .join(
        scb_lazy_lf_level4,
        left_on=["year", "code_4"],
        right_on=["year", "ssyk_code"],
        how="left",
    )

In [17]:
inspect_lazy(daioe_scb_years)

Rows: 4,686
Columns: 28


## 9) Inspect Unmatched DAIOE Codes

Identify DAIOE rows that have no SCB SSYK4 match.


In [18]:
# DAIOE codes with no SCB match
daioe_scb_years_unmatched = daioe_lazy_lf_extended\
    .join(
        scb_lazy_lf_level4,
        left_on=["year", "code_4"],
        right_on=["year", "ssyk_code"],
        how="anti",
    )



In [19]:
inspect_lazy(daioe_scb_years_unmatched)

Rows: 0
Columns: 27


## 10) Post-Merge Validation

Run quick schema and shape checks on the filtered joined frame before aggregation.


In [20]:
daioe_scb_years.collect_schema()

Schema([('year', Int64),
        ('daioe_allapps', Float64),
        ('daioe_stratgames', Float64),
        ('daioe_videogames', Float64),
        ('daioe_imgrec', Float64),
        ('daioe_imgcompr', Float64),
        ('daioe_imggen', Float64),
        ('daioe_readcompr', Float64),
        ('daioe_lngmod', Float64),
        ('daioe_translat', Float64),
        ('daioe_speechrec', Float64),
        ('daioe_genai', Float64),
        ('pctl_rank_allapps', Float64),
        ('pctl_rank_stratgames', Float64),
        ('pctl_rank_videogames', Float64),
        ('pctl_rank_imgrec', Float64),
        ('pctl_rank_imgcompr', Float64),
        ('pctl_rank_imggen', Float64),
        ('pctl_rank_readcompr', Float64),
        ('pctl_rank_lngmod', Float64),
        ('pctl_rank_translat', Float64),
        ('pctl_rank_speechrec', Float64),
        ('pctl_rank_genai', Float64),
        ('code_1', String),
        ('code_2', String),
        ('code_3', String),
        ('code_4', String),
        ('tot

In [21]:
daioe_scb_years.collect_schema().names()

['year',
 'daioe_allapps',
 'daioe_stratgames',
 'daioe_videogames',
 'daioe_imgrec',
 'daioe_imgcompr',
 'daioe_imggen',
 'daioe_readcompr',
 'daioe_lngmod',
 'daioe_translat',
 'daioe_speechrec',
 'daioe_genai',
 'pctl_rank_allapps',
 'pctl_rank_stratgames',
 'pctl_rank_videogames',
 'pctl_rank_imgrec',
 'pctl_rank_imgcompr',
 'pctl_rank_imggen',
 'pctl_rank_readcompr',
 'pctl_rank_lngmod',
 'pctl_rank_translat',
 'pctl_rank_speechrec',
 'pctl_rank_genai',
 'code_1',
 'code_2',
 'code_3',
 'code_4',
 'total_count']

In [22]:
inspect_lazy(daioe_scb_years)

Rows: 4,686
Columns: 28


## 11) Generalized Aggregation + Percentiles

Aggregate SSYK4-1 with a reusable function and add within-year percentiles.

In [29]:
def aggregate_daioe_level(  # noqa: PLR0913
    lf: pl.LazyFrame,
    code_col: str,
    level_label: str,
    *,
    weight_col: str = "total_count",
    prefix: str = "daioe_",
    add_percentiles: bool = True,
    pct_scale: int = 100,
    descending: bool = False,
) -> pl.LazyFrame:

    daioe_cols = [c for c in lf.collect_schema().names() if c.startswith(prefix)]
    w = pl.col(weight_col)

    out = (
        lf
        .group_by(["year", code_col])
        .agg(
            w.sum().alias("weight_sum"),
            pl.col(daioe_cols).mean().name.suffix("_avg"),
            ((pl.col(daioe_cols) * w).sum() / w.sum()).name.suffix("_wavg"),
        )
        .with_columns(pl.lit(level_label).alias("level"))
        .rename({code_col: "ssyk_code"})
    )

    if not add_percentiles:
        return out

    group_keys = ["year", "level"]

    rank_expr = (
        pl.col(f"^{prefix}.*_(avg|wavg)$")
        .rank(method="average", descending=descending)
        .over(group_keys)
    )

    n_expr = pl.len().over(group_keys)

    return out.with_columns(
        (
            pl.when(n_expr > 1)
            .then((rank_expr - 1) / (n_expr - 1))
            .otherwise(0.0)
            * pct_scale
        ).name.prefix("pctl_"),
    )

In [30]:
levels = {
    "code_4": "SSYK4",
    "code_3": "SSYK3",
    "code_2": "SSYK2",
    "code_1": "SSYK1",
}

aggregated = [
    aggregate_daioe_level(daioe_scb_years, col, label)
    for col, label in levels.items()
]

daioe_all_levels = (
    pl.concat(aggregated)
    .sort(["level", "year", "ssyk_code"])
)


In [31]:
inspect_lazy(daioe_all_levels)

Rows: 6,853
Columns: 48


In [32]:
print(
    daioe_all_levels
    .group_by("level")
    .len()
    .collect(),
)


shape: (4, 2)
┌───────┬──────┐
│ level ┆ len  │
│ ---   ┆ ---  │
│ str   ┆ u32  │
╞═══════╪══════╡
│ SSYK3 ┆ 1595 │
│ SSYK4 ┆ 4686 │
│ SSYK1 ┆ 99   │
│ SSYK2 ┆ 473  │
└───────┴──────┘


## 12) Build 1-5 Level Exposure Columns

Create `daioe_<index>_Level_Exposure` columns from weighted percentile ranks (`pctl_daioe_*_wavg`) using quintile-style bins.

In [33]:
# Convert weighted percentile ranks (0..100) into 1-5 exposure levels
pct_cols = [
    c
    for c in daioe_all_levels.collect_schema().names()
    if c.startswith("pctl_daioe_") and c.endswith("_wavg")
]

_q1, _q2, _q3, _q4 = QUINTILE_BOUNDS

exposure_exprs = []
for col_name in pct_cols:
    metric = col_name[len("pctl_daioe_"):-len("_wavg")]
    out_col = f"daioe_{metric}_Level_Exposure"
    p = pl.col(col_name)

    exposure_exprs.append(
        pl.when(p.is_null())
        .then(None)
        .when(p <= _q1)
        .then(1)
        .when(p <= _q2)
        .then(2)
        .when(p <= _q3)
        .then(3)
        .when(p <= _q4)
        .then(4)
        .otherwise(5)
        .cast(pl.Int8)
        .alias(out_col),
    )

daioe_all_levels = daioe_all_levels.with_columns(exposure_exprs)

print([c for c in daioe_all_levels.collect_schema().names() if c.endswith("_Level_Exposure")])

['daioe_allapps_Level_Exposure', 'daioe_stratgames_Level_Exposure', 'daioe_videogames_Level_Exposure', 'daioe_imgrec_Level_Exposure', 'daioe_imgcompr_Level_Exposure', 'daioe_imggen_Level_Exposure', 'daioe_readcompr_Level_Exposure', 'daioe_lngmod_Level_Exposure', 'daioe_translat_Level_Exposure', 'daioe_speechrec_Level_Exposure', 'daioe_genai_Level_Exposure']


## 13) Pre-Merge Diagnostics

Inspect schemas and frame shapes before final integration.

In [34]:
daioe_all_levels.collect_schema()

Schema([('year', Int64),
        ('ssyk_code', String),
        ('weight_sum', Int64),
        ('daioe_allapps_avg', Float64),
        ('daioe_stratgames_avg', Float64),
        ('daioe_videogames_avg', Float64),
        ('daioe_imgrec_avg', Float64),
        ('daioe_imgcompr_avg', Float64),
        ('daioe_imggen_avg', Float64),
        ('daioe_readcompr_avg', Float64),
        ('daioe_lngmod_avg', Float64),
        ('daioe_translat_avg', Float64),
        ('daioe_speechrec_avg', Float64),
        ('daioe_genai_avg', Float64),
        ('daioe_allapps_wavg', Float64),
        ('daioe_stratgames_wavg', Float64),
        ('daioe_videogames_wavg', Float64),
        ('daioe_imgrec_wavg', Float64),
        ('daioe_imgcompr_wavg', Float64),
        ('daioe_imggen_wavg', Float64),
        ('daioe_readcompr_wavg', Float64),
        ('daioe_lngmod_wavg', Float64),
        ('daioe_translat_wavg', Float64),
        ('daioe_speechrec_wavg', Float64),
        ('daioe_genai_wavg', Float64),
        

In [35]:
scb_lazy_lf.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('age', String),
        ('sex', String),
        ('year', Int64),
        ('count', Int64),
        ('occupation', String)])

In [39]:
inspect_lazy(daioe_all_levels)

Rows: 6,853
Columns: 59


## 14) Final Merge

Attach DAIOE aggregates back to the SCB base table.

In [ ]:
final_merge = scb_lazy_lf\
    .join(
        daioe_all_levels,
        on=["year", "ssyk_code", "level"],
        how="left",
    )

In [41]:
inspect_lazy(final_merge)

Rows: 137,060
Columns: 65


In [42]:
#dd = final_merge.limit(30).collect()

## 15) Final Merge — Inspect

Quick shape and schema check before computing changes.

In [43]:
inspect_lazy(final_merge)

Rows: 137,060
Columns: 65


---
## 16) Compute 1-Yr, 3-Yr and 5-Yr Employment Changes

Compute absolute and percent changes in `count` over 1, 3, and 5 years using window shifts over `(level, ssyk_code, age, sex)` groups, then join the features back onto the full dataset.

In [44]:
keys = ["level", "ssyk_code", "age", "sex"]

changes_lf = (
    final_merge
    .group_by([*keys, "year"])
    .agg(pl.col("count").sum().alias("emp_count"))
    .sort([*keys, "year"])
    .with_columns(
        # Guard: replace zero prior-year counts with null before dividing.
        # A group going from 0 → N has no meaningful percentage baseline,
        # so null is more honest than inf (which would propagate into
        # downstream aggregations and plots).
        pl.col("emp_count").shift(1).over(keys).replace(0, None).alias("_prev1"),
        pl.col("emp_count").shift(3).over(keys).replace(0, None).alias("_prev3"),
        pl.col("emp_count").shift(5).over(keys).replace(0, None).alias("_prev5"),
    )
    .with_columns(
        (pl.col("emp_count") - pl.col("_prev1")).alias("chg_1y"),
        (pl.col("emp_count") - pl.col("_prev3")).alias("chg_3y"),
        (pl.col("emp_count") - pl.col("_prev5")).alias("chg_5y"),
        ((pl.col("emp_count") / pl.col("_prev1") - 1) * 100).alias("pct_chg_1y"),
        ((pl.col("emp_count") / pl.col("_prev3") - 1) * 100).alias("pct_chg_3y"),
        ((pl.col("emp_count") / pl.col("_prev5") - 1) * 100).alias("pct_chg_5y"),
    )
    .drop("emp_count", "_prev1", "_prev3", "_prev5")
)

inspect_lazy(changes_lf)

Rows: 137,060
Columns: 11


## 17) Merge Changes + Reorder Columns

Join the change metrics back onto the full dataset and bring key columns to the front.

In [45]:
join_cols = ["year", "level", "ssyk_code", "age", "sex"]

processed_lf = final_merge.join(changes_lf, on=join_cols, how="left")

first_cols = [
    "year", "level", "ssyk_code", "occupation",
    "sex", "age", "age_group",
    "count", "weight_sum",
    "chg_1y", "chg_3y", "chg_5y",
    "pct_chg_1y", "pct_chg_3y", "pct_chg_5y",
]

schema_cols = list(processed_lf.collect_schema())
other_cols = [c for c in schema_cols if c not in first_cols]

processed_lf = (
    processed_lf
    .select(first_cols + other_cols)
    .sort(join_cols, descending=True, nulls_last=True)
)

processed_lf.limit(5).collect()

year,level,ssyk_code,occupation,sex,age,age_group,count,weight_sum,chg_1y,chg_3y,chg_5y,pct_chg_1y,pct_chg_3y,pct_chg_5y,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,level_right,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure
i64,str,str,str,str,str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
2024,"""SSYK4""","""9629""","""Other service workers not else…","""women""","""60-64""","""Senior (50+)""",2159,30744,131,275,426,6.459566,14.596603,24.58165,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""men""","""60-64""","""Senior (50+)""",2640,30744,43,222,350,1.655757,9.181141,15.283843,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""women""","""55-59""","""Senior (50+)""",2067,30744,-101,-70,-71,-4.658672,-3.27562,-3.320861,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,2,2,4,3,2,2,2,2,2,2,2
2024,"""SSYK4""","""9629""","""Other service workers not else…","""men""","""55-59""","""Senior (50+)""",2437,30744,-6,-101,-72,-0.2456,-3.979511,-2.869669,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,28.339077,0.240595,5.527369,0.402341,0.192295,0.527986,0.22591,0.568319,0.024523,0.523234,1.801313,"""SSYK4""",33.176471,28.0,78.823529,41.647059,35.294118,26.352941,20.941176,23.294118,24.941176,26.588235,23.058824,33.176471,28.0,78.823529,41.647059,35.294118,2

## 18) Export Processed Dataset

Write the final dataset with change features to parquet for downstream use.

In [46]:
processed_path = DATA_DIR / "daioe_scb_years_processed.parquet"

processed_lf.sink_parquet(processed_path)
print(f"Saved: {processed_path}")

Saved: /run/media/joseph-data/DataDisk/1_Work/RA_Work/Rebuilding/AI_Econ_daioe_years_v2/data/daioe_scb_years_processed.parquet
